In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
import datetime as dt

In [0]:
dbutils.widgets.text("login", "mariiasirska")
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.dropdown("reset", "true", ["true", "false"])

In [0]:
login = dbutils.widgets.get("login")
catalog = dbutils.widgets.get("catalog")
bronze = f"{catalog}.{login}_bronze"
silver = f"{catalog}.{login}_silver"

In [0]:
SRC = f"{bronze}.olist_customers_cdc_src"
CUR = f"{silver}.olist_customers_cdc_current"
LOG = f"{silver}.olist_customers_cdc_changelog"
STATE = f"{silver}.lab10_cdc_state"
VIEW = f"{silver}.olist_customers_cdc_scd2"
BIZ_COLS = ["customer_id", "customer_unique_id", "zip_code_prefix", "city", "state"]

In [0]:
if dbutils.widgets.get("reset") == "true":
    for t in [SRC, CUR, LOG, STATE]:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    spark.sql(f"DROP VIEW IF EXISTS {VIEW}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SRC} (
  customer_id STRING, customer_unique_id STRING, zip_code_prefix STRING, city STRING, state STRING
) USING DELTA
COMMENT 'Simulated operational customers table. Change Data Feed is the CDC source'
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CUR} (
  customer_id STRING NOT NULL, customer_unique_id STRING, zip_code_prefix STRING, city STRING, state STRING,
  _src_version BIGINT COMMENT 'Source commit version that produced this row (guards against stale events)',
  _src_commit_ts TIMESTAMP,
  _loaded_at TIMESTAMP
) USING DELTA COMMENT 'Latest state of every customer, maintained incrementally by MERGE'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG} (
  customer_id STRING, customer_unique_id STRING, zip_code_prefix STRING, city STRING, state STRING,
  change_type STRING, _commit_version BIGINT, _commit_timestamp TIMESTAMP
) USING DELTA COMMENT 'Append-only change history: one row per change event (insert / update_postimage / delete)'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {STATE} (source_table STRING, last_version BIGINT, updated_at TIMESTAMP) USING DELTA
""")
if spark.table(STATE).filter(F.col("source_table") == SRC).count() == 0:
    spark.sql(f"INSERT INTO {STATE} VALUES ('{SRC}', -1, current_timestamp())")


In [0]:
def latest_per_key(changes):
    w = Window.partitionBy("customer_id").orderBy(F.col("_commit_version").desc())
    return (
        changes.filter("_change_type <> 'update_preimage'")
        .withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn")
    )


def apply_to_current(latest):
    set_map = {c: f"s.{c}" for c in BIZ_COLS if c != "customer_id"}
    set_map.update({"_src_version": "s._commit_version", "_src_commit_ts": "s._commit_timestamp", "_loaded_at": "current_timestamp()"})
    ins_map = {"customer_id": "s.customer_id", **set_map}
    (
        DeltaTable.forName(spark, CUR).alias("t")
        .merge(latest.alias("s"), "t.customer_id = s.customer_id")
        .whenMatchedDelete(condition="s._change_type = 'delete' AND s._commit_version > t._src_version")
        .whenMatchedUpdate(condition="s._change_type <> 'delete' AND s._commit_version > t._src_version", set=set_map)
        .whenNotMatchedInsert(condition="s._change_type <> 'delete'", values=ins_map)   # also covers a missed insert
        .execute()
    )


def append_changelog(changes):
    ev = (
        changes.filter("_change_type <> 'update_preimage'")
        .select(*BIZ_COLS, F.col("_change_type").alias("change_type"), "_commit_version", "_commit_timestamp")
    )
    (
        DeltaTable.forName(spark, LOG).alias("t")
        .merge(ev.alias("s"), "t.customer_id = s.customer_id AND t._commit_version = s._commit_version")
        .whenNotMatchedInsertAll()
        .execute()
    )


In [0]:
def get_bookmark():
    return spark.table(STATE).filter(F.col("source_table") == SRC).first()["last_version"]


def set_bookmark(v):
    spark.sql(f"UPDATE {STATE} SET last_version = {int(v)}, updated_at = current_timestamp() WHERE source_table = '{SRC}'")

In [0]:
def process_changes():
    last = get_bookmark()
    latest_v = spark.sql(f"DESCRIBE HISTORY {SRC}").agg(F.max("version")).first()[0]
    if latest_v <= last:
        print(f"no new versions (bookmark={last}, source at v{latest_v})")
        return {"from_version": last + 1, "to_version": latest_v, "events": 0}
    changes = spark.sql(f"SELECT * FROM table_changes('{SRC}', {last + 1}, {latest_v})")
    by_type = {r["_change_type"]: r["count"] for r in changes.groupBy("_change_type").count().collect()}
    append_changelog(changes)
    apply_to_current(latest_per_key(changes))
    set_bookmark(latest_v)
    out = {"from_version": last + 1, "to_version": latest_v, "events": sum(by_type.values()), "by_type": by_type}
    print(out)
    return out

In [0]:
def verify():
    a = spark.table(SRC).select(*BIZ_COLS)
    b = spark.table(CUR).select(*BIZ_COLS)
    diff = a.exceptAll(b).count() + b.exceptAll(a).count()
    print(f"source rows = {a.count()}, current rows = {b.count()}, differing rows = {diff}")
    return diff

In [0]:
spark.sql(f"""
INSERT INTO {SRC}
SELECT customer_id, customer_unique_id, zip_code_prefix, city, state
FROM {silver}.olist_customers_silver
ORDER BY customer_id
LIMIT 1000
""")
first_run = process_changes()
assert verify() == 0

Simulate operational changes: updates, deletes, insert:

In [0]:
spark.sql(f"UPDATE {SRC} SET city = concat(city, ' (moved)'), state = 'SP' WHERE pmod(hash(customer_id), 20) = 0")
spark.sql(f"DELETE FROM {SRC} WHERE pmod(hash(customer_id), 50) = 1")
spark.sql(f"""
INSERT INTO {SRC}
SELECT customer_id, customer_unique_id, zip_code_prefix, city, state
FROM {silver}.olist_customers_silver s
WHERE NOT EXISTS (SELECT 1 FROM {SRC} t WHERE t.customer_id = s.customer_id)
ORDER BY customer_id
LIMIT 30
""")
spark.sql(f"UPDATE {SRC} SET zip_code_prefix = '99999' WHERE pmod(hash(customer_id), 200) = 0")

In [0]:
last = get_bookmark()
ev = spark.sql(f"SELECT * FROM table_changes('{SRC}', {last + 1})")
display(ev.groupBy("_commit_version", "_change_type").count().orderBy("_commit_version", "_change_type"))

In [0]:
n_source = spark.table(SRC).count()
second_run = process_changes()
print(f"full table has {n_source} rows, the incremental run read {second_run['events']} change events")
assert verify() == 0

display(spark.sql(f"""
SELECT change_type, count(*) AS events, count(DISTINCT customer_id) AS customers
FROM {LOG} WHERE _commit_version >= {second_run['from_version']}
GROUP BY change_type ORDER BY change_type
"""))

In [0]:
before = {"current": spark.table(CUR).count(), "changelog": spark.table(LOG).count()}
set_bookmark(-1)
process_changes()
after = {"current": spark.table(CUR).count(), "changelog": spark.table(LOG).count()}
print("before:", before)
print("after: ", after)
assert before == after, "reprocessing must be a no-op"
assert verify() == 0
print("IDEMPOTENT: re-reading all versions changed nothing")

In [0]:
victim = spark.table(CUR).filter("_src_version >= 2").first()
print("customer:", victim["customer_id"], "| city now:", victim["city"], "| held version:", victim["_src_version"])

late = spark.createDataFrame(
    [(victim["customer_id"], "stale-id", "00000", "STALE CITY", "XX", "update_postimage", 1, dt.datetime(2020, 1, 1))],
    "customer_id string, customer_unique_id string, zip_code_prefix string, city string, state string, "
    "_change_type string, _commit_version long, _commit_timestamp timestamp",
)
apply_to_current(late)

row = spark.table(CUR).filter(F.col("customer_id") == victim["customer_id"]).first()
assert row["city"] == victim["city"] and row["state"] == victim["state"], "stale event overwrote newer data"
print("late event ignored, city is still:", row["city"])

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {VIEW} AS
SELECT customer_id, customer_unique_id, zip_code_prefix, city, state, change_type,
       _commit_version AS version,
       _commit_timestamp AS valid_from,
       lead(_commit_timestamp) OVER (PARTITION BY customer_id ORDER BY _commit_version) AS valid_to,
       (lead(_commit_version) OVER (PARTITION BY customer_id ORDER BY _commit_version) IS NULL
         AND change_type <> 'delete') AS is_current
FROM {LOG}
""")

multi = spark.table(LOG).groupBy("customer_id").count().orderBy(F.desc("count")).first()["customer_id"]
print("customer changed several times:", multi)
display(spark.table(VIEW).filter(F.col("customer_id") == multi).orderBy("version"))

gone = spark.table(LOG).filter("change_type = 'delete'").orderBy(F.desc("_commit_version")).first()["customer_id"]
spark.sql(f"DELETE FROM {SRC} WHERE customer_id = '{gone}'")
process_changes()

print("deleted customer (still in history, absent from current):", gone)
display(spark.table(VIEW).filter(F.col("customer_id") == gone).orderBy("version"))
assert spark.table(CUR).filter(F.col("customer_id") == gone).count() == 0

In [0]:
summary = spark.sql(f"""
SELECT
  (SELECT count(*) FROM {SRC}) AS source_rows,
  (SELECT count(*) FROM {CUR}) AS current_rows,
  (SELECT count(*) FROM {LOG}) AS changelog_events,
  (SELECT count(*) FROM {LOG} WHERE change_type = 'delete') AS deletes_captured,
  (SELECT count(*) FROM {VIEW} WHERE is_current) AS scd2_current_rows
""")
display(summary)
display(spark.sql(f"DESCRIBE HISTORY {CUR}").selectExpr(
    "version", "timestamp", "operation",
    "operationMetrics['numTargetRowsInserted'] AS inserted",
    "operationMetrics['numTargetRowsUpdated'] AS updated",
    "operationMetrics['numTargetRowsDeleted'] AS deleted").orderBy("version"))
print("first run:", first_run)
print("second run:", second_run)

Batch ingestion repeatedly reads and overwrites entire source tables, which increases compute overhead as data grows and loses intermediate state transitions and hard deletes. Using Delta Change Data Feed with a version bookmark processes only new commits, capturing insert, update_postimage, and delete events incrementally.

Slowly Changing Dimensions: Combining CDC events with MERGE INTO supports both SCD Type 1 and SCD Type 2. The SCD2 view preserves full audit history even after a record is deleted in the source.

Late-Arriving Data: Out-of-order or delayed events can corrupt target tables if an older update overwrites a newer state. 